In [2]:
# Species Information Retrieval - T5 Fine-Tuning with Custom QA Dataset (with Dynamic Wikipedia Retrieval)

#  STEP 1: Install Required Libraries
!pip install transformers datasets evaluate rouge_score requests

# Disable W&B tracking to avoid API key prompts
import os
os.environ["WANDB_DISABLED"] = "true"

# STEP 2: Load Dataset
import pandas as pd
from datasets import Dataset

df = pd.read_csv("Generated_QA_Dataset_Updated.csv")  # Update path if needed
dataset = Dataset.from_pandas(df)
train_test = dataset.train_test_split(test_size=0.2)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=0c89d1559d9566e2ecb558b4497eb7c9e3953782f5358b7d9b2ac143e0966fd6
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [3]:
#  STEP 3: Preprocess & Tokenize
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-base")

def preprocess(example):
    input_text = f"question: {example['question']}  context: {example['context']}"
    target_text = example["answer"]
    model_inputs = tokenizer(input_text, max_length=512, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(target_text, max_length=64, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = train_test.map(preprocess)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/95 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/24 [00:00<?, ? examples/s]

In [4]:
#  STEP 4: Load and Train the Model
from transformers import T5ForConditionalGeneration, Trainer, TrainingArguments
import torch

model = T5ForConditionalGeneration.from_pretrained("t5-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
tokenizer.model_input_names = ["input_ids", "attention_mask", "labels"]

training_args = TrainingArguments(
    output_dir="./species-t5-finetuned",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=15,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=500,
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
)

trainer.train()


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-4-2717717942.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
10,18.283400
20,8.833400
30,3.495600
40,1.317400
50,0.742100
60,0.503900
70,0.364500
80,0.247200
90,0.248700
100,0.166900


TrainOutput(global_step=360, training_loss=1.0176529101199574, metrics={'train_runtime': 234.8579, 'train_samples_per_second': 6.067, 'train_steps_per_second': 1.533, 'total_flos': 867764994048000.0, 'train_loss': 1.0176529101199574, 'epoch': 15.0})

In [4]:
#  STEP 5: Evaluate
from collections import Counter
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# Custom token-level F1
def compute_f1(prediction, ground_truth):
    pred_tokens = prediction.lower().split()
    gt_tokens = ground_truth.lower().split()
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

def evaluate_model(dataset):
    predictions, references, f1_scores = [], [], []
    model.eval()
    for item in dataset:
        input_text = f"question: {item['question']}  context: {item['context']}"
        input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            output = model.generate(input_ids.to(device), max_length=64)
        pred = tokenizer.decode(output[0], skip_special_tokens=True)
        ref = item["answer"]
        predictions.append(pred)
        references.append(ref)
        f1_scores.append(compute_f1(pred, ref))
    print("BLEU:", bleu.compute(predictions=predictions, references=[[r] for r in references]))
    print("ROUGE:", rouge.compute(predictions=predictions, references=references))
    print("F1 (avg):", sum(f1_scores) / len(f1_scores))

# Run evaluation
evaluate_model(train_test["test"])





BLEU: {'bleu': 0.7188971322361775, 'precisions': [0.9037037037037037, 0.8378378378378378, 0.7912087912087912, 0.76], 'brevity_penalty': 0.8751733190429475, 'length_ratio': 0.8823529411764706, 'translation_length': 135, 'reference_length': 153}
ROUGE: {'rouge1': np.float64(0.9173611111111111), 'rouge2': np.float64(0.7362082362082362), 'rougeL': np.float64(0.9173611111111111), 'rougeLsum': np.float64(0.9180555555555555)}
F1 (avg): 0.9173611111111111


In [5]:
!pip install wikipedia-api


  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia-api: filename=Wikipedia_API-0.8.1-py3-none-any.whl size=15383 sha256=1f879ec319ecb7a4511f225a1e657a2e5542e6bd670e3adf011e87bfba5762fb
  Stored in directory: /root/.cache/pip/wheels/0b/0f/39/e8214ec038ccd5aeb8c82b957289f2f3ab2251febeae5c2860
Successfully built wikipedia-api


In [6]:
#  STEP 6: Dynamic Wikipedia Context Retrieval
import wikipediaapi

# Set custom user agent per Wikipedia policy
wiki = wikipediaapi.Wikipedia(language="en", user_agent="SpeciesQA/1.0 (your_email@example.com)")

def detect_section_from_question(question):
    question = question.lower()

    if any(q in question for q in ["eat", "diet", "food"]):
        return "Diet"
    elif any(q in question for q in ["live", "habitat", "location", "found", "range"]):
        return "Habitat"
    elif any(q in question for q in ["behavior", "behaviour", "social", "act"]):
        return "Behaviour"
    elif any(q in question for q in ["reproduce", "reproduction", "birth", "offspring"]):
        return "Reproduction"
    elif any(q in question for q in ["threat", "danger", "extinct", "predator", "vulnerable"]):
        return "Threats"
    elif any(q in question for q in ["kind of species", "what species", "type of animal", "category"]):
        return "Summary"
    elif any(q in question for q in ["where does it", "locate", "found", "native to"]):
        return "Habitat"
    elif any(q in question for q in ["stable", "population", "conservation status"]):
        return "Conservation"
    elif any(q in question for q in ["rare", "scarce", "uncommon"]):
        return "Conservation"
    else:
        return ""

def get_species_context(species_name, question):
    section = detect_section_from_question(question)
    page = wiki.page(species_name)
    if not page.exists():
        return "No Wikipedia content found."
    for s in page.sections:
        if section and s.title.lower() == section.lower():
            return s.text
    return page.summary  # fallback

def generate_dynamic_answer(question, species_name):
    context = get_species_context(species_name, question)
    print("\n📘 Context:", context)
    input_text = f"question: {question}  context: {context}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)
    output = model.generate(input_ids, max_length=64)
    return tokenizer.decode(output[0], skip_special_tokens=True)


In [8]:
#  STEP 7: Save Model (Optional)
model.save_pretrained("species-t5-finetuned")
tokenizer.save_pretrained("species-t5-finetuned")

# 🙋‍♂️ STEP 8: User Input Interface
print("\n🔍 Ask the model a species-related question!")
user_question = input("Enter your question: ")
user_species = input("Enter the species name (e.g., Panthera leo): ")

user_answer = generate_dynamic_answer(user_question, user_species)
print("\n🤖 Answer:", user_answer)



🔍 Ask the model a species-related question!
Enter your question: where does it live?
Enter the species name (e.g., Panthera leo): Anartia jatrophae

📘 Context: Anartia jatrophae, the white peacock, is a species of butterfly found in the southeastern United States, Central America, and throughout much of South America. The white peacock's larval hosts are water hyssop (Bacopa monnieri),  lemon bacopa (Bacopa caroliniensis), tropical waterhyssop (Bacopa innominata), frogfruit (Phyla nodiflora), lanceleaf frogfruit (Phyla lanceolata), and Carolina wild petunia (Ruellia caroliniana).
The males of the species display a unique territorial behavior, in which they stake out a territory typically 15 meters in diameter that contains larval host plants. They perch in this area and aggressively protect it from other insects and other male white peacocks.

🤖 Answer: southeastern United States, Central America, and throughout much of South America


In [9]:
import shutil
shutil.make_archive("species-t5-finetuned", 'zip', "species-t5-finetuned")


'/content/species-t5-finetuned.zip'